# 強化学習のコード

### TD(0)
状態価値 V(s) を学ぶ予測問題

### n-step TD
同じく V(s) だが、数歩先の報酬をまとめて使う

### Q学習
行動価値 Q(s,a) を学んで最適行動を取りたい制御問題

### n-step Q学習
Q学習を数歩先に伸ばしたもの

### SARSA
実際に辿る行動系列に沿って Q(s,a) を学ぶ制御問題

### n-step SARSA
SARSA を数歩先まで伸ばしたもの

### SARSA(λ)
eligibility trace を使って、過去の訪問状態行動にも一気に誤差を流す

全部 NumPy だけで動く。
一つずつ独立しているので、コピペで試せる。

## TD(0)
問題設定
7マスの通路があり、左端が崖で報酬0、右端がゴールで報酬1。
エージェントは中央から始まり、左右にランダムに歩く。
ここでは行動を学ばない。欲しいのは「各状態から将来どれくらい右端に行けそうか」という V(s)。

In [ ]:
import numpy as np

np.random.seed(0)

N = 7
TERMINALS = [0, N - 1]
START = N // 2

alpha = 0.1
gamma = 1.0
episodes = 200

V = np.zeros(N)
V[N - 1] = 1.0

def step(state):
    action = np.random.choice([-1, 1])
    next_state = state + action
    reward = 1.0 if next_state == N - 1 else 0.0
    done = next_state in TERMINALS
    return next_state, reward, done

for episode in range(episodes):
    s = START
    while True:
        s_next, r, done = step(s)
        V[s] += alpha * (r + gamma * V[s_next] - V[s])
        s = s_next
        if done:
            break

print("TD(0) value estimates")
for i, v in enumerate(V):
    print(f"state {i}: {v:.3f}")

このコードでは、学習対象は V(s) だけ。
右端に近い状態ほど 1 に近づき、左側は 0 に寄る。

## n-step TD
問題設定
今度は10マスの通路で、右端に着いたときだけ報酬1。
行動はやはりランダムで、予測だけやる。
違いは、1歩先で更新せず、3歩分まとめて戻す点にある。

In [ ]:
import numpy as np

np.random.seed(1)

N = 10
TERMINALS = [0, N - 1]
START = N // 2

alpha = 0.1
gamma = 1.0
n = 3
episodes = 300

V = np.zeros(N)
V[N - 1] = 1.0

def step(state):
    action = np.random.choice([-1, 1])
    next_state = state + action
    reward = 1.0 if next_state == N - 1 else 0.0
    done = next_state in TERMINALS
    return next_state, reward, done

for episode in range(episodes):
    states = [START]
    rewards = [0.0]
    T = float("inf")
    t = 0

    while True:
        if t < T:
            s = states[t]
            s_next, r, done = step(s)
            states.append(s_next)
            rewards.append(r)
            if done:
                T = t + 1

        tau = t - n + 1
        if tau >= 0:
            G = 0.0
            upper = min(tau + n, int(T))
            for i in range(tau + 1, upper + 1):
                G += (gamma ** (i - tau - 1)) * rewards[i]
            if tau + n < T:
                G += (gamma ** n) * V[states[tau + n]]

            s_tau = states[tau]
            if s_tau not in TERMINALS:
                V[s_tau] += alpha * (G - V[s_tau])

        if tau == T - 1:
            break
        t += 1

print("3-step TD value estimates")
for i, v in enumerate(V):
    print(f"state {i}: {v:.3f}")

ここでは 3 ステップ分の報酬が一気に戻るので、TD(0) より情報の伝播が速く見える。
学習対象はやはり V(s) のまま。

## Q学習
問題設定
4×4 のグリッドで、左上から右下のゴールに向かう。
報酬は1手ごとに -1、ゴール到達で 0。
最短経路を学ばせたいので、行動価値 Q(s,a) を持つ。

In [ ]:
import numpy as np

np.random.seed(2)

H, W = 4, 4
ACTIONS = [0, 1, 2, 3]  # up, down, left, right
START = (0, 0)
GOAL = (3, 3)

alpha = 0.5
gamma = 0.95
epsilon = 0.1
episodes = 500

Q = np.zeros((H, W, len(ACTIONS)))

def step(state, action):
    r, c = state
    if action == 0:
        r = max(0, r - 1)
    elif action == 1:
        r = min(H - 1, r + 1)
    elif action == 2:
        c = max(0, c - 1)
    else:
        c = min(W - 1, c + 1)

    next_state = (r, c)
    done = next_state == GOAL
    reward = 0.0 if done else -1.0
    return next_state, reward, done

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))

for episode in range(episodes):
    s = START
    for _ in range(100):
        a = choose_action(s)
        s_next, r, done = step(s, a)

        sr, sc = s
        nr, nc = s_next
        td_target = r + gamma * np.max(Q[nr, nc])
        Q[sr, sc, a] += alpha * (td_target - Q[sr, sc, a])

        s = s_next
        if done:
            break

policy_symbols = ["↑", "↓", "←", "→"]
print("Q-learning greedy policy")
for i in range(H):
    row = []
    for j in range(W):
        if (i, j) == GOAL:
            row.append("G")
        else:
            row.append(policy_symbols[int(np.argmax(Q[i, j]))])
    print(" ".join(row))

ここでの学習対象は Q(s,a)。
更新式では次状態の最大値を使うので、実際にどう動いたかより「そこから最善ならどうなるか」を見ている。

## n-step Q学習
問題設定
同じ 4×4 グリッドだが、今度は 3 手先までの報酬をまとめて使う。
最後だけブートストラップで max Q を足す。

In [ ]:
import numpy as np

np.random.seed(3)

H, W = 4, 4
ACTIONS = [0, 1, 2, 3]
START = (0, 0)
GOAL = (3, 3)

alpha = 0.3
gamma = 0.95
epsilon = 0.1
n = 3
episodes = 500

Q = np.zeros((H, W, len(ACTIONS)))

def step(state, action):
    r, c = state
    if action == 0:
        r = max(0, r - 1)
    elif action == 1:
        r = min(H - 1, r + 1)
    elif action == 2:
        c = max(0, c - 1)
    else:
        c = min(W - 1, c + 1)

    next_state = (r, c)
    done = next_state == GOAL
    reward = 0.0 if done else -1.0
    return next_state, reward, done

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))

for episode in range(episodes):
    states = [START]
    actions = []
    rewards = [0.0]
    T = float("inf")
    t = 0

    while True:
        if t < T:
            s = states[t]
            a = choose_action(s)
            actions.append(a)

            s_next, r, done = step(s, a)
            states.append(s_next)
            rewards.append(r)

            if done:
                T = t + 1

        tau = t - n + 1
        if tau >= 0:
            G = 0.0
            upper = min(tau + n, int(T))
            for i in range(tau + 1, upper + 1):
                G += (gamma ** (i - tau - 1)) * rewards[i]

            if tau + n < T:
                nr, nc = states[tau + n]
                G += (gamma ** n) * np.max(Q[nr, nc])

            s_tau = states[tau]
            if s_tau != GOAL:
                sr, sc = s_tau
                a_tau = actions[tau]
                Q[sr, sc, a_tau] += alpha * (G - Q[sr, sc, a_tau])

        if tau == T - 1:
            break
        t += 1

policy_symbols = ["↑", "↓", "←", "→"]
print("3-step Q-learning greedy policy")
for i in range(H):
    row = []
    for j in range(W):
        if (i, j) == GOAL:
            row.append("G")
        else:
            row.append(policy_symbols[int(np.argmax(Q[i, j]))])
    print(" ".join(row))

これは見た目こそ Q学習に近いが、1手ごとの誤差ではなく、3手ぶんのリターンを一気に使っている。
遠い報酬の影響が少し早く手前まで届く。

## SARSA
問題設定
崖歩き問題にする。
4×12 の盤面で、下段の途中は全部崖。
スタートは左下、ゴールは右下。崖に落ちると -100 でスタートへ戻る。
ここでは「今の方策で実際に取りうる危なさ」込みで学びたいので SARSA が合う。

In [ ]:
import numpy as np

np.random.seed(4)

H, W = 4, 12
ACTIONS = [0, 1, 2, 3]
START = (3, 0)
GOAL = (3, 11)
CLIFF = {(3, c) for c in range(1, 11)}

alpha = 0.5
gamma = 1.0
epsilon = 0.1
episodes = 500

Q = np.zeros((H, W, len(ACTIONS)))

def step(state, action):
    r, c = state
    if action == 0:
        r = max(0, r - 1)
    elif action == 1:
        r = min(H - 1, r + 1)
    elif action == 2:
        c = max(0, c - 1)
    else:
        c = min(W - 1, c + 1)

    next_state = (r, c)

    if next_state in CLIFF:
        return START, -100.0, False

    if next_state == GOAL:
        return next_state, 0.0, True

    return next_state, -1.0, False

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))

for episode in range(episodes):
    s = START
    a = choose_action(s)

    for _ in range(1000):
        s_next, r, done = step(s, a)
        a_next = choose_action(s_next)

        sr, sc = s
        nr, nc = s_next
        td_target = r + gamma * Q[nr, nc, a_next]
        Q[sr, sc, a] += alpha * (td_target - Q[sr, sc, a])

        s, a = s_next, a_next
        if done:
            break

policy_symbols = ["↑", "↓", "←", "→"]
print("SARSA greedy policy")
for i in range(H):
    row = []
    for j in range(W):
        pos = (i, j)
        if pos == START:
            row.append("S")
        elif pos == GOAL:
            row.append("G")
        elif pos in CLIFF:
            row.append("C")
        else:
            row.append(policy_symbols[int(np.argmax(Q[i, j]))])
    print(" ".join(row))

この問題では、Q学習だと崖の縁ギリギリを通ることが多い。
SARSA は ε-greedy のランダムな踏み外し込みで値を作るので、上の段を回る安全寄りの経路を選びやすい。

## n-step SARSA
問題設定
同じ崖歩きだが、今度は 4 ステップ先まで見て更新する。
実際に選んだ行動列に沿う点は SARSA のまま。

In [ ]:
import numpy as np

np.random.seed(5)

H, W = 4, 12
ACTIONS = [0, 1, 2, 3]
START = (3, 0)
GOAL = (3, 11)
CLIFF = {(3, c) for c in range(1, 11)}

alpha = 0.3
gamma = 1.0
epsilon = 0.1
n = 4
episodes = 500

Q = np.zeros((H, W, len(ACTIONS)))

def step(state, action):
    r, c = state
    if action == 0:
        r = max(0, r - 1)
    elif action == 1:
        r = min(H - 1, r + 1)
    elif action == 2:
        c = max(0, c - 1)
    else:
        c = min(W - 1, c + 1)

    next_state = (r, c)

    if next_state in CLIFF:
        return START, -100.0, False

    if next_state == GOAL:
        return next_state, 0.0, True

    return next_state, -1.0, False

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))

for episode in range(episodes):
    states = [START]
    actions = [choose_action(START)]
    rewards = [0.0]
    T = float("inf")
    t = 0

    while True:
        if t < T:
            s = states[t]
            a = actions[t]

            s_next, r, done = step(s, a)
            states.append(s_next)
            rewards.append(r)

            if done:
                T = t + 1
            else:
                actions.append(choose_action(s_next))

        tau = t - n + 1
        if tau >= 0:
            G = 0.0
            upper = min(tau + n, int(T))
            for i in range(tau + 1, upper + 1):
                G += (gamma ** (i - tau - 1)) * rewards[i]

            if tau + n < T:
                nr, nc = states[tau + n]
                a_boot = actions[tau + n]
                G += (gamma ** n) * Q[nr, nc, a_boot]

            s_tau = states[tau]
            sr, sc = s_tau
            a_tau = actions[tau]
            Q[sr, sc, a_tau] += alpha * (G - Q[sr, sc, a_tau])

        if tau == T - 1:
            break
        t += 1

policy_symbols = ["↑", "↓", "←", "→"]
print("4-step SARSA greedy policy")
for i in range(H):
    row = []
    for j in range(W):
        pos = (i, j)
        if pos == START:
            row.append("S")
        elif pos == GOAL:
            row.append("G")
        elif pos in CLIFF:
            row.append("C")
        else:
            row.append(policy_symbols[int(np.argmax(Q[i, j]))])
    print(" ".join(row))

ここではブートストラップの相手が max ではなく、実際に取る a になっている。
だから「その方策で4手進んだらどうなるか」がそのまま戻る。

## SARSA(λ)
問題設定
5×5 の迷路で、中央に壁を置く。
1歩ごとに -1、ゴールで 0。
ここでは eligibility trace の効き方を見たいので、後方観測の代表として SARSA(λ) を使う。
1回の TD 誤差で、今だけでなく、少し前に通った状態行動にも更新が入る。

In [ ]:
import numpy as np

np.random.seed(6)

H, W = 5, 5
ACTIONS = [0, 1, 2, 3]
START = (0, 0)
GOAL = (4, 4)
WALLS = {(1, 2), (2, 2), (3, 2)}

alpha = 0.2
gamma = 0.95
epsilon = 0.1
lambda_ = 0.8
episodes = 600

Q = np.zeros((H, W, len(ACTIONS)))

def step(state, action):
    r, c = state
    nr, nc = r, c

    if action == 0:
        nr = max(0, r - 1)
    elif action == 1:
        nr = min(H - 1, r + 1)
    elif action == 2:
        nc = max(0, c - 1)
    else:
        nc = min(W - 1, c + 1)

    if (nr, nc) in WALLS:
        nr, nc = r, c

    next_state = (nr, nc)
    done = next_state == GOAL
    reward = 0.0 if done else -1.0
    return next_state, reward, done

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    r, c = state
    return int(np.argmax(Q[r, c]))

for episode in range(episodes):
    E = np.zeros_like(Q)

    s = START
    a = choose_action(s)

    for _ in range(300):
        s_next, r, done = step(s, a)
        a_next = choose_action(s_next)

        sr, sc = s
        nr, nc = s_next

        target = r if done else r + gamma * Q[nr, nc, a_next]
        delta = target - Q[sr, sc, a]

        E[sr, sc, a] += 1.0
        Q += alpha * delta * E
        E *= gamma * lambda_

        s, a = s_next, a_next
        if done:
            break

policy_symbols = ["↑", "↓", "←", "→"]
print("SARSA(lambda) greedy policy")
for i in range(H):
    row = []
    for j in range(W):
        pos = (i, j)
        if pos == START:
            row.append("S")
        elif pos == GOAL:
            row.append("G")
        elif pos in WALLS:
            row.append("W")
        else:
            row.append(policy_symbols[int(np.argmax(Q[i, j]))])
    print(" ".join(row))

このコードの核心はここにある。
```Python
E[sr, sc, a] += 1.0
Q += alpha * delta * E
E *= gamma * lambda_
```

直前の一個だけでなく、履歴に残っている全部の状態行動に対して、delta が濃淡つきで流れる。
λ を 0 にすると、痕跡は即消えるので、普通の 1-step SARSA に近づく。
0.8 くらいにすると、5手前くらいまで目に見えて効く。

## 見方の整理

TD(0) で学ぶのは V(s)。
行動は固定されていて、「この状態はどれくらい良さそうか」だけを推定する。

n-step TD も対象は V(s) のまま。
ただし 1 手先ではなく n 手先までの報酬をまとめて使う。

Q学習で学ぶのは Q(s,a)。
「次は最善を打つ」と仮定して値を作るので、オフポリシーになる。

SARSA も Q(s,a) を学ぶ。
ただし更新先は max ではなく、実際に選んだ次の行動 a′ なので、今の方策の危なさや癖がそのまま値に入る。

n-step 版は、これらを数歩先まで伸ばしたもの。
違いは最後のブートストラップ先だけで、Q学習なら max、SARSA なら実際の a を使う。

eligibility trace は、対象そのものを変えるというより、誤差の流し方を変える。
1手ずつ戻す代わりに、最近通った履歴全体へまとめて更新をかける。

## 最初に触る順番

実際に手元で試すなら、順番はかなり大事。
TD(0) を先に動かすと、V(s) しか持たない予測の感触が掴める。
そのあと Q学習に行くと、「行動」が増えたせいで表の次元が一つ増えたのが見える。
SARSA を試すと、max を消しただけなのに経路が変わる。
最後に SARSA(λ) を回すと、trace の役割がはっきり出る。

一番おすすめなのは、各コードに 5 行だけ追加して、各エピソードの経路を出すこと。
たとえば Q学習や SARSA なら、episode 0 では壁や崖にぶつかって遠回りし、episode 300 あたりでは最短や安全経路に収束していく。
その変化を見ると、式より先に腹落ちする。